In [ ]:
# ================================================
# 03_muestreo.ipynb
# Muestreo estratificado + verificación de reproducibilidad
# Proyecto: Estilometría de Noticias — Visualización de Datos, USFQ
# Autor: Steeven Quezada
#
# Entrada:  data/processed/dataset_preprocessed.csv  (38,473 filas, salida de 02_preprocesamiento.ipynb)
# Salida:   data/processed/dataset_muestra.csv       (3,000 filas: 1,500 falsas / 1,500 verdaderas)
# ================================================

import pandas as pd
import numpy as np
from scipy import stats

# ------------------------------------------------
# PASO 1: Cargar el dataset preprocesado completo
# ------------------------------------------------
df = pd.read_csv('../data/processed/dataset_preprocessed.csv')

print(df.columns.tolist())        # confirma que 'label' existe como columna normal
print(df['label'].value_counts())  # confirma las clases antes de muestrear

# ------------------------------------------------
# PASO 2: Muestreo estratificado (1,500 por clase, semilla fija)
# ------------------------------------------------
n_por_clase = 1500

fake = df[df['label'] == 0].sample(n=min(n_por_clase, (df['label'] == 0).sum()), random_state=42)
real = df[df['label'] == 1].sample(n=min(n_por_clase, (df['label'] == 1).sum()), random_state=42)

muestra = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)

# Corrección del parseo de fechas mixtas ('Jan' vs 'January') detectado durante la validación
muestra['date_parsed'] = pd.to_datetime(muestra['date'], format='mixed', errors='coerce')
muestra['month'] = muestra['date_parsed'].dt.to_period('M').astype(str)

muestra.to_csv('../data/processed/dataset_muestra.csv', index=False, quoting=1)

print(muestra.shape)
print(muestra['label'].value_counts())


# ------------------------------------------------
# PASO 3: Verificación de reproducibilidad — Cohen's d y Mann-Whitney
# Los 5 valores deben coincidir con los declarados en la propuesta:
#   title_length=1.48 | questions=0.71 | avg_word_len=-0.58 | exclamations=0.54 | text_length≈0.19-0.20
# ------------------------------------------------

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx - 1) * x.std(ddof=1) ** 2 + (ny - 1) * y.std(ddof=1) ** 2) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled_std

variables = ['title_length', 'questions', 'avg_word_len', 'exclamations', 'text_length']

print(f"\n{'Variable':<18}{'Cohen d':>10}{'Mann-Whitney p':>18}")
for v in variables:
    fake_vals = muestra[muestra['label'] == 0][v]
    real_vals = muestra[muestra['label'] == 1][v]
    d = cohens_d(fake_vals, real_vals)
    _, p = stats.mannwhitneyu(fake_vals, real_vals)
    print(f"{v:<18}{d:>10.2f}{p:>18.4g}")

C:\Users\Steeven\AppData\Local\Temp\ipykernel_16792\133732397.py:18: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/dataset_preprocessed.csv')


['title', 'text', 'subject', 'date', 'label', 'text_length', 'title_length', 'date_parsed', 'month', 'exclamations', 'questions', 'caps_ratio', 'avg_word_len', 'text_clean', 'title_clean', 'clean_length', 'title_text_clean']
label
1    21191
0    17282
Name: count, dtype: int64
(3000, 17)
label
1    1500
0    1500
Name: count, dtype: int64

Variable             Cohen d    Mann-Whitney p
title_length            1.48        4.326e-279
questions               0.71        1.739e-141
avg_word_len           -0.58         9.373e-96
exclamations            0.54         1.884e-81
text_length             0.19         3.999e-09
